In [2]:
# Round Robin CPU Scheduling Simulator
# Operating Systems Assignment - Refactored Version

from google.colab import output
output.enable_custom_widget_manager()

import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from collections import deque
import pandas as pd

# ---------------- USER INTERFACE INPUT CONTROLS ----------------

activity_input = widgets.Dropdown(
    options=[
        ("Activity 1 - No Context Switch Cost", "activity1"),
        ("Activity 2 - 1 ms Context Switch Cost", "activity2")
    ],
    value="activity1",
    description="Activity:"
)

process_input = widgets.Text(
    value="5,3,4,2",
    description="Burst times:",
    placeholder="Example: 5,3,4,2"
)

arrival_input = widgets.Text(
    value="0,1,2,3",
    description="Arrival times:"
)

quantum_input = widgets.IntSlider(
    value=2,
    min=1,
    max=10,
    step=1,
    description="Time quantum:"
)

context_input = widgets.IntSlider(
    value=0,
    min=0,
    max=5,
    step=1,
    description="Switch cost:"
)

run_button = widgets.Button(
    description="Run Simulation",
    button_style="success",
    icon="play"
)

play_button = widgets.Play(
    value=0,
    min=0,
    max=1,
    step=1,
    interval=800
)

step_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=1,
    step=1,
    description="Step:",
    continuous_update=False
)

next_button = widgets.Button(
    description="Next Step",
    button_style="info",
    icon="step-forward"
)

reset_button = widgets.Button(
    description="Reset",
    button_style="warning",
    icon="refresh"
)

chart_output = widgets.Output()
result_output = widgets.Output()
status_output = widgets.Output()

timeline_segments = []
simulation_results = []
elapsed_total_time = 0


# ---------------- PRESET CONFIGURATIONS ----------------

def handle_activity_preset(change=None):
    if activity_input.value == "activity1":
        process_input.value = "5,3,4,2"
        arrival_input.value = "0,1,2,3"
        quantum_input.value = 2
        context_input.value = 0
    else:
        process_input.value = "5,4,3,2"
        arrival_input.value = "0,1,2,3"
        quantum_input.value = 2
        context_input.value = 1

activity_input.observe(handle_activity_preset, names="value")


# ---------------- SCHEDULING CORE ENGINE ----------------

def compute_round_robin_schedule(bursts, arrivals, quantum, switch_cost):
    cpu_bursts_left = bursts.copy()
    num_processes = len(bursts)

    first_start = [None] * num_processes
    completion = [0] * num_processes
    segments = []
    ready_q = deque()
    is_added = [False] * num_processes

    current_tick = 0
    finished_count = 0
    prev_process = None
    switch_count = 0

    while finished_count < num_processes:

        # Queue up arrived processes
        for idx in range(num_processes):
            if arrivals[idx] <= current_tick and not is_added[idx]:
                ready_q.append(idx)
                is_added[idx] = True

        # Handle idle CPU time by jumping to next arrival
        if not ready_q:
            future_arrivals = [
                arrivals[idx] for idx in range(num_processes)
                if not is_added[idx]
            ]

            if not future_arrivals:
                break

            current_tick = min(future_arrivals)
            continue

        curr_proc = ready_q.popleft()

        # Apply context switch penalty if switching between distinct processes
        if (
            prev_process is not None
            and prev_process != curr_proc
            and switch_cost > 0
        ):
            segments.append({
                "process": "CS",
                "start": current_tick,
                "end": current_tick + switch_cost
            })

            switch_count += 1
            current_tick += switch_cost

            # Check for new process arrivals during context switch overhead
            for idx in range(num_processes):
                if arrivals[idx] <= current_tick and not is_added[idx]:
                    ready_q.append(idx)
                    is_added[idx] = True

        if first_start[curr_proc] is None:
            first_start[curr_proc] = current_tick

        segment_start = current_tick
        execution_slice = min(quantum, cpu_bursts_left[curr_proc])

        current_tick += execution_slice
        cpu_bursts_left[curr_proc] -= execution_slice

        segments.append({
            "process": f"P{curr_proc + 1}",
            "start": segment_start,
            "end": current_tick
        })

        # Enqueue newly arrived processes after execution slice
        for idx in range(num_processes):
            if arrivals[idx] <= current_tick and not is_added[idx]:
                ready_q.append(idx)
                is_added[idx] = True

        if cpu_bursts_left[curr_proc] > 0:
            ready_q.append(curr_proc)
        else:
            completion[curr_proc] = current_tick
            finished_count += 1

        prev_process = curr_proc

    # Gather metrics per process
    process_metrics = []
    for idx in range(num_processes):
        tat = completion[idx] - arrivals[idx]
        wt = tat - bursts[idx]
        rt = first_start[idx] - arrivals[idx]

        process_metrics.append({
            "Process": f"P{idx + 1}",
            "AT": arrivals[idx],
            "BT": bursts[idx],
            "CT": completion[idx],
            "TAT": tat,
            "WT": wt,
            "RT": rt
        })

    return segments, process_metrics, switch_count


# ---------------- GANTT CHART VISUALIZER ----------------

def render_gantt_chart(step):
    with chart_output:
        clear_output(wait=True)

        if not timeline_segments:
            print("Click Run Simulation first.")
            return

        fig, ax = plt.subplots(figsize=(11, 4))

        visible_segs = timeline_segments[:step]

        process_names = sorted(
            set(
                s["process"] for s in timeline_segments
                if s["process"] != "CS"
            ),
            key=lambda name: int(name[1:])
        )

        row_indices = {name: i for i, name in enumerate(process_names)}

        if any(s["process"] == "CS" for s in timeline_segments):
            row_indices["CS"] = len(process_names)

        for seg in visible_segs:
            p_name = seg["process"]
            start_time = seg["start"]
            span_width = seg["end"] - seg["start"]
            y_pos = row_indices[p_name]

            ax.barh(
                y_pos,
                span_width,
                left=start_time,
                height=0.55,
                edgecolor="black",
                alpha=0.85
            )

            ax.text(
                start_time + span_width / 2,
                y_pos,
                p_name,
                ha="center",
                va="center",
                fontsize=9
            )

        active_time = visible_segs[-1]["end"] if visible_segs else 0

        ax.axvline(
            active_time,
            linestyle="--",
            linewidth=2,
            label=f"Time = {active_time}"
        )

        ax.set_yticks(list(row_indices.values()))
        ax.set_yticklabels(list(row_indices.keys()))
        ax.set_xlabel("CPU Time (ms)")
        ax.set_ylabel("Process")
        ax.set_title("Round Robin Gantt Chart")
        ax.set_xlim(0, max(elapsed_total_time, 1))
        ax.set_xticks(range(int(elapsed_total_time) + 1))
        ax.grid(axis="x", alpha=0.3)
        ax.legend()

        plt.tight_layout()
        plt.show()
        plt.close(fig)

        print(
            f"Current time: {active_time} ms | "
            f"Steps: {step}/{len(timeline_segments)}"
        )


# ---------------- INTERACTIVE WIDGET CONTROLS ----------------

def handle_step_update(change):
    if timeline_segments:
        render_gantt_chart(change["new"])

step_slider.observe(handle_step_update, names="value")

def handle_play_state_change(change):
    if timeline_segments and step_slider.value != change["new"]:
        step_slider.value = change["new"]

play_button.observe(handle_play_state_change, names="value")

def handle_next_step(button):
    if timeline_segments:
        step_slider.value = min(
            step_slider.value + 1,
            len(timeline_segments)
        )

next_button.on_click(handle_next_step)

def handle_reset(button):
    play_button.value = 0
    step_slider.value = 0
    render_gantt_chart(0)

reset_button.on_click(handle_reset)


# ---------------- SIMULATION EXECUTION HANDLER ----------------

def handle_run_simulation(button):
    global timeline_segments, simulation_results, elapsed_total_time

    try:
        bursts = [
            int(item.strip())
            for item in process_input.value.split(",")
        ]

        arrivals = [
            int(item.strip())
            for item in arrival_input.value.split(",")
        ]

        if not bursts or any(val <= 0 for val in bursts):
            raise ValueError(
                "Burst times must be positive integers."
            )

        if len(bursts) != len(arrivals):
            raise ValueError(
                "Enter one arrival time per process."
            )

        if any(val < 0 for val in arrivals):
            raise ValueError(
                "Arrival times cannot be negative."
            )

        quantum = quantum_input.value
        switch_cost = context_input.value

        timeline_segments, simulation_results, total_switches = compute_round_robin_schedule(
            bursts,
            arrivals,
            quantum,
            switch_cost
        )

        elapsed_total_time = max(
            (seg["end"] for seg in timeline_segments),
            default=0
        )

        play_button.value = 0
        step_slider.max = len(timeline_segments)
        step_slider.value = 0
        play_button.max = len(timeline_segments)

        results_df = pd.DataFrame(simulation_results)

        active_cpu_time = sum(bursts)
        total_overhead = total_switches * switch_cost

        cpu_efficiency = (
            active_cpu_time / elapsed_total_time * 100
            if elapsed_total_time > 0 else 0
        )

        with result_output:
            clear_output(wait=True)

            print("PROCESS SCHEDULING RESULTS")
            display(results_df)

            print(
                "\nAverage Waiting Time:",
                round(results_df["WT"].mean(), 2), "ms"
            )

            print(
                "Average Turnaround Time:",
                round(results_df["TAT"].mean(), 2), "ms"
            )

            print(
                "Average Response Time:",
                round(results_df["RT"].mean(), 2), "ms"
            )

            print("Context Switches:", total_switches)
            print("Total Switch Overhead:", total_overhead, "ms")
            print("Total Elapsed Time:", elapsed_total_time, "ms")
            print(
                "CPU Efficiency:",
                round(cpu_efficiency, 2), "%"
            )

        with status_output:
            clear_output(wait=True)
            print(
                "Simulation ready! Use Play or Next Step."
            )

        render_gantt_chart(0)

    except Exception as err:
        with status_output:
            clear_output(wait=True)
            print("Input error:", err)

run_button.on_click(handle_run_simulation)


# ---------------- DISPLAY DASHBOARD LAYOUT ----------------

display(widgets.VBox([
    widgets.HTML(
        "<h3>Round Robin CPU Scheduling Simulator</h3>"
    ),
    activity_input,
    process_input,
    arrival_input,
    quantum_input,
    context_input,
    run_button,
    widgets.HTML("<b>Animation Controls</b>"),
    widgets.HBox([
        play_button,
        next_button,
        reset_button
    ]),
    step_slider,
    status_output,
    chart_output,
    result_output
]))